<a href="https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/OpenEnv_gpt_oss_(20B)_Reinforcement_Learning_2048_Game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <img width="35" height="35" alt="image" src="https://github.com/user-attachments/assets/2700a971-e5d6-4036-b03f-2f89c9791609" /> OpenEnv: Agentic Execution Environments
We're using the new [OpenEnv](https://github.com/meta-pytorch/OpenEnv) library which has over 2000+ environments for RL!

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install).

# Goal: Make Qwen3-4B play games with Reinforcement Learning

Our goal is to make OpenAI's open model Qwen3-4B play the wordle game with reinforcement learning. We want the model to devise a strategy to play wordle, and we will run this strategy until we win or lose.

<img width="270" height="265" alt="image" src="https://github.com/user-attachments/assets/1a51a172-ff1a-40a2-8bee-2746f5017aa0" />

# Installation
We'll be using [Unsloth](https://github.com/unslothai/unsloth) to do RL on GPT-OSS 20B, and [OpenEnv](https://github.com/meta-pytorch/OpenEnv) for the environment interactions. Unsloth saves 70% VRAM usage and makes reinforcement learning 2 to 6x faster!

In [1]:
# %%capture
# import os, importlib.util
# !pip install --upgrade -qqq uv
# if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
#     try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
#     except: get_numpy = "numpy"
#     !uv pip install -qqq \
#         "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers==4.56.2" trackio \
#         "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
#         "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
#         git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
# elif importlib.util.find_spec("unsloth") is None:
#     !uv pip install -qqq unsloth trackio
# !uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

We will then install [OpenEnv](https://github.com/meta-pytorch/OpenEnv) from source:

In [2]:
# %%capture
# !pip install -qqq fastapi uvicorn requests open_spiel
# !pip install fastapi uvicorn requests
# !pip install open_spiel --prefer-binary
# # !git clone https://github.com/meta-pytorch/OpenEnv.git > /dev/null 2>&1
# # %cd OpenEnv
# # !git checkout 83dda10
# import subprocess, sys, os
# from pathlib import Path
# sys.path.insert(0, '.')  # Add OpenEnv root for envs module
# sys.path.insert(0, './src')
# working_directory = str(Path.cwd().parent.absolute() / "OpenEnv")

We'll load Qwen3-4B and set some parameters:
* `max_seq_length = 768` The maximum context length of the model. Increasing it will use more memory.
* `lora_rank = 32` The larger this number, the smarter the RL process, but the slower and more memory usage. 

In [3]:
import os
# os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ['UNSLOTH_VLLM_STANDBY_UTIL_OVERRIDE']='1'
os.environ['CUDA_VISIBLE_DEVICES']='7'
os.environ['UNSLOTH_ENABLE_LOGGING']='1'
! rm -rf unsloth_compiled_cache
import torch
import torch._dynamo

torch._dynamo.reset()

In [4]:
import os
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Can increase for longer RL output
lora_rank = 32       # Larger rank = smarter, but slower
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-4B",
    load_in_4bit = False,
    max_seq_length = max_seq_length,
    fast_inference = True,
)

[unsloth.import_fixes|INFO]Unsloth: ROCm detection did not match any known hints.
[unsloth.import_fixes|INFO]Unsloth: Patching protobuf.MessageFactory.GetPrototype
[unsloth.import_fixes|INFO]Unsloth: fbgemm_gpu_genai==1.5.0+cu130 detected.
[unsloth.import_fixes|INFO]Unsloth: torch==2.10.0+cu130 and torchvision==0.25.0+cu130 are compatible.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/datta0/.venvs/pyenv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
[unsloth.import_fixes|INFO]Unsloth: SM100 (NVIDIA B200) detected but vLLM 0.16.1.1 should include PDL fix - skipping workaround
[unsloth.import_fixes|INFO]Unsloth: Patched triton CompiledKernel with num_ctas/cluster_dims for torch.compile compatibility.
[unsloth.import_fixes|INFO]Unsloth: Patched torch.nn.init.trunc_normal_ for fp16/bf16 stability.
[unsloth.import_fixes|INFO]Unsloth: Patched enable_input_require_grads for vision model compatibility
[unsloth_zoo.log|INFO]Unsloth: set VIDEO_TOTAL_PIXELS: 90316800
[unsloth_zoo.log|INFO]Unsloth: Patched Gemma3Processor.__call__.
[unsloth_zoo.log|INFO]Unsloth: Patched Gemma3RMSNorm.forward.
[unsloth_zoo.log|INFO]Unsloth: Patched Gemma3Attention.forward.
[unsloth_zoo.log|INFO]Unsloth: Patched AutoHfQuantizer.merge_qua

Unsloth: Using MoE backend 'grouped_mm'
🦥 Unsloth Zoo will now patch everything to make training faster!


TMA benchmarks will be running without grid constant TMA descriptor.
/home/datta0/.venvs/pyenv/lib/python3.11/site-packages/triton/runtime/autotuner.py:101: DeprecationWarning: warmup, rep, and use_cuda_graph parameters are deprecated. See https://github.com/triton-lang/triton/pull/4496 for details.
  warnings.warn(("warmup, rep, and use_cuda_graph parameters are deprecated. See "
[unsloth_zoo.log|INFO]Unsloth: Could not find torchao.prototype.blockwise_fp8_inference.blockwise_quantization.blockwise_fp8_gemm


...

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 102/102 [00:12<00:00,  7.89it/s]
Capturing CUDA graphs (decode, FULL):  89%|████████▊ | 62/70 [00:03<00:00, 17.40it/s]2026-03-10 10:32:50,972 - INFO - cubin_loader.py:209 - flashinfer.jit: Fetching cubin e86f0e45764555d070c3d143b4caaea61a45b777/fmha/trtllm-gen//fmhaSm100fKernel_QkvBfloat16OBfloat16H128PagedKvCausalP16MultiCtasKvCgaVarSeqQ8Kv128StaticSwapsAbForGen.cubin from https://edge.urm.nvidia.com/artifactory/sw-kernelinferencelibrary-public-generic-local/e86f0e45764555d070c3d143b4caaea61a45b777/fmha/trtllm-gen/fmhaSm100fKernel_QkvBfloat16OBfloat16H128PagedKvCausalP16MultiCtasKvCgaVarSeqQ8Kv128StaticSwapsAbForGen.cubin
2026-03-10 10:32:50,973 - INFO - cubin_loader.py:81 - flashinfer.jit: Acquired lock for unsloth_compiled_cache/.cache/flashinfer/cubins/e86f0e45764555d070c3d143b4caaea61a45b777/fmha/trtllm-gen/fmhaSm100fKernel_QkvBfloat16OBfloat16H128PagedKvCausalP16MultiCtasKvCgaVarSeqQ8Kv128StaticSwapsAbForGe

INFO 03-10 10:32:51 [gpu_model_runner.py:5360] Graph capturing finished in 17 secs, took 0.44 GiB
INFO 03-10 10:32:51 [vllm_utils.py:736] Unsloth: Patched vLLM v1 graph capture finished in 17 secs.


INFO 03-10 10:32:52 [core.py:282] init engine (profile, create kv cache, warmup model) took 105.78 seconds
INFO 03-10 10:32:53 [llm.py:388] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['q_norm', 'post_attention_layernorm', 'attention_norm', 'k_norm', 'ffn_norm', 'post_layernorm', 'input_layernorm', 'norm2', 'post_feedforward_layernorm', 'layer_norm1', 'pre_feedforward_layernorm', 'layer_norm2', 'norm1', 'norm']


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
✅ Copied 3122 attributes (including 0 config-related dicts)
📋 Skipped 10 non-config dictionaries
⏭️ Skipped 20 total attributes (tensors, modules, non-config dicts, etc.)
    Sample: ['all_tied_weights_keys (dict not in config)', 'can_record_outputs (dict not in config)', 'dtype', 'dummy_inputs (dict not in config)', 'is_gradient_checkpointing']... and 15 more
Unsloth: Just some info: will skip parsing ['q_norm', 'post_attention_layernorm', 'attention_norm', 'k_norm', 'ffn_norm', 'post_layernorm', 'input_layernorm', 'norm2', 'cross_attn_input_layernorm', 'cross_attn_post_attention_layernorm', 'post_feedforward_layernorm', 'layer_norm1', 'pre_feedforward_layernorm', 'layer_norm2', 'norm1', 'norm']
unsloth/Qwen3-4B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


To do efficient RL, we will use [LoRA](https://arxiv.org/abs/2106.09685), which allows us to only add 1 to 5% of extra weights to the model for finetuning purposes. This allows us to save memory usage by over 60%, and yet it retains good accuracy. Read Unsloth's [GPT-OSS RL Guide](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning) for more details.

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## Initialize the Environment

Let's begin by setting up the environment that will be used during training.  
For this task, we'll rely on the **TextArena** environment from **OpenEnv**, which exposes a familiar Gymnasium-style API (`reset()`, `step()`, etc.) to simplify interaction.

In this example, we'll connect to the hosted environment at [openenv/wordle](https://huggingface.co/spaces/openenv/wordle).  
For production use or custom configurations, we **strongly recommend** running the environment locally via Docker. The hosted versions on the Hub currently have limited concurrency support, so duplicating the Space to your own account is the preferred approach in those cases.

For more information, refer to the [TRL-OpenEnv documentation](https://huggingface.co/docs/trl/main/en/openenv).


In [6]:
from huggingface_hub import notebook_login

notebook_login()

User is already logged in.
[huggingface_hub._login|INFO]User is already logged in.


In [7]:
from textarena_env import TextArenaEnv

wordle_url = "https://openenv-wordle.hf.space" # Duplicate the Space and update this!
env = TextArenaEnv(base_url=wordle_url)
# wordle_url = "openenv/wordle"
# env = TextArenaEnv.from_hub(repo_id=wordle_url)

## Rollout function with helpers

The **rollout function** defines how the agent interacts with the environment during GRPO training.
It's responsible for generating model completions, collecting feedback (rewards), and returning all necessary information for optimization.

In this setup:
- The function is called automatically by the **GRPOTrainer** during each training step.  
- It uses the trainer's built-in `generate_rollout_completions()` method for efficient generation with vLLM in colocate mode.
- Each rollout represents a full interaction loop. The model guesses, receives feedback from Wordle, and updates based on reward signals.
- The **`env_mask`** tracks which tokens are model-generated vs environment-generated, ensuring only model tokens contribute to the training loss.

The rewards track different aspects of the agent's performance. Helper functions (like `rollout_once`) handle one episode of interaction, keeping the main `rollout_func` clean and modular.

This modular approach allows GRPO to efficiently sample, evaluate, and improve the model's guessing strategy through reinforcement learning.

First, we define the `system_prompt` that guides the model's behavior as an expert Wordle solver with strategic reasoning and structured responses.

In [8]:
# @title System prompt (click to expand)
system_prompt = """
You are an expert Wordle solver with deep knowledge of English vocabulary, letter frequency patterns, and optimal guessing strategies.

## GAME RULES

1. The target is a 5-letter English word
2. You have 6 attempts to guess the correct word
3. After each guess, you receive color-coded feedback:
   - GREEN: Letter is correct and in the correct position
   - YELLOW: Letter is in the word but in the wrong position
   - GRAY: Letter is not in the word at all
4. All guesses must be valid 5-letter English words
5. You cannot reuse a word you've already guessed

## RESPONSE FORMAT

Only respond with your next guess in square brackets, e.g., [crane].

Format:
```
[guess]
```


## STRATEGIC APPROACH

Do not repeat the same guess twice.

### Opening Strategy
- Start with words rich in common vowels (A, E, I, O, U) and consonants (R, S, T, L, N)
- Optimal starters: CRANE, SLATE, STARE, AROSE, IRATE
- Prioritize words that test the most common letters in different positions

### Mid-Game Strategy
- Use confirmed GREEN letters in their correct positions
- Place YELLOW letters in different positions than where they appeared
- Eliminate GRAY letters entirely from consideration
- If multiple letters are unknown, prioritize common letter combinations (TH, CH, ST, ER, etc.)
- Consider letter frequency: E is most common, followed by A, R, I, O, T, N, S

### Vowel Placement
- Most 5-letter words have 2 vowels
- Common patterns: vowel-consonant-vowel (like CRANE) or consonant-vowel-vowel-consonant-vowel (like QUEUE)
- If you have 1-2 vowels confirmed, consider where the others might be

### Advanced Tactics
- Use "sacrificial" guesses to test multiple new letters if you have attempts to spare
- Avoid repeating letter patterns unless you're certain (e.g., SPEED has two E's)
- Think about word endings: -ER, -LY, -ED, -ING are common but may not fit the 5-letter constraint
- Consider less common letters (Q, X, Z, J) only when you've eliminated most common options

### Common Pitfalls to Avoid
- Don't reuse X letters
- Don't place Y letters in the same position they appeared
- Don't ignore confirmed G letters
- Don't guess words that contradict known information

## EXAMPLES

### Example 1: Opening Guess
"Starting with a word that tests common vowels and consonants in varied positions."
[crane]

### Example 2: After Receiving Feedback
Previous guess: CRANE
Feedback: C=gray, R=yellow, A=green, N=gray, E=yellow

"A is confirmed in position 2. R and E are in the word but need different positions. C and N are eliminated. I'll try a word with A in position 2, and test R and E in new positions along with common letters like S and T."
[spare]

### Example 3: Narrowing Down
Previous guesses: CRANE (C=gray, R=yellow, A=green, N=gray, E=yellow), SPARE (S=gray, P=gray, A=green, R=green, E=green)
Feedback summary: _ARE_ with R in position 4, A in position 2, E in position 5

"I have _AR E_ confirmed. Position 1 and 3 are unknown. Common letters to try: T, L, D, B, F, G. Testing with TARED."
[tared]

### Example 4: Final Deduction
Previous feedback shows: _ARED with position 1 unknown and all common consonants tested

"Only position 1 remains. I've eliminated S, P, C, N. Common starting consonants left are B, F, G, H. BARED is a common word."
[bared]

## LETTER FREQUENCY REFERENCE

Most common letters in 5-letter words (in order):
S, E, A, O, R, I, L, T, N, U, D, Y, C, P, M, H, G, B, K, F

Most common starting letters:
S, C, B, T, P, A, F, G, D, M

Most common ending letters:
E, Y, T, S, R, L, N, D

## IMPORTANT CONSTRAINTS

- Use lowercase only
- One guess per response
- Must be exactly 5 letters
- Must be a real English word from standard dictionaries
- Never repeat a previous guess
- Always include brief reasoning before your guess

## YOUR GOAL

Solve the Wordle in as few guesses as possible by strategically using feedback to eliminate impossible words and narrow down the solution space efficiently.
"""

Now, let's define the `rollout_func`:

This function orchestrates the interaction between the model and the Wordle environment. For each prompt in the batch, it runs the episode interaction, collecting rewards and model outputs for GRPO optimization.

In [9]:
max_new_tokens = 8
max_turns = 6

def rollout_func(prompts, trainer):
    """
    Rollout function for GRPO training with environment interaction.

    This function is called by GRPOTrainer to generate completions and compute rewards.
    It uses trainer.generate_rollout_completions() for inference.

    Args:
        prompts: List of prompts to generate from
        trainer: GRPOTrainer instance containing context and configuration

    Returns:
        Dictionary with prompt_ids, completion_ids, logprobs, env_mask, and reward signals
    """
    episode_prompt_ids = []
    episode_completion_ids = []
    episode_logprobs = []
    episode_env_masks = []
    correctness_rewards = []
    position_rewards = []
    format_rewards = []

    for prompt_text in prompts:
        episode = rollout_once(
            trainer=trainer,
            env=env,
            tokenizer=tokenizer,
            dataset_prompt=prompt_text,
            system_prompt=system_prompt,
            max_turns=max_turns,
            max_new_tokens=max_new_tokens,
        )
        episode_prompt_ids.append(episode["prompt_ids"])
        episode_completion_ids.append(episode["completion_ids"])
        episode_logprobs.append(episode["logprobs"])
        episode_env_masks.append(episode["env_mask"])
        correctness_rewards.append(episode["correct_reward"])
        position_rewards.append(episode["position_reward"])
        format_rewards.append(compute_format_reward(episode["model_outputs"]))

    return {
        "prompt_ids": episode_prompt_ids,
        "completion_ids": episode_completion_ids,
        "logprobs": episode_logprobs,
        "env_mask": episode_env_masks,
        "correct_reward": correctness_rewards,
        "position_reward": position_rewards,
        "format_reward": format_rewards,
    }

### Define `rollout_once`

The `rollout_once` function runs **one full interaction loop** between the model and the Wordle environment using the trainer's generation method.  
It executes a mini episode of gameplay, from generating a guess to receiving and processing feedback.

Here's the step-by-step breakdown:

1. **Environment reset:** Start a new game session and initialize the observation.  
2. **Prompt construction:** Combine the system prompt, current state, and user messages to form the model input.  
3. **Generation:** Use `trl.experimental.openenv.generate_rollout_completions()` to produce the model's guess efficiently.  
4. **Feedback extraction:** Parse the environment's response using helpers like `extract_guess()` and `extract_wordle_feedback()`.  
5. **Reward calculation:** Compute rewards based on correctness, green/yellow feedback, and repetition penalty.
6. **Return structured rollout data:** Includes prompt/completion IDs, logprobs, `env_mask`, and all computed reward components.

**Important: The `env_mask` mechanism**

In multi-turn environments like Wordle, the completion includes both:
- **Model-generated tokens** (the guesses): These should contribute to the loss during training.
- **Environment feedback tokens** (game responses): These should NOT contribute to the loss.

The `env_mask` is a list of 1s and 0s that marks which tokens are model-generated (`1`) vs environment-generated (`0`).  
The GRPOTrainer uses this mask to exclude environment tokens from the loss calculation, ensuring the model only learns from its own outputs.

This modular design ensures that each episode can be processed independently while still providing rich feedback for the **GRPO training loop**.

In [10]:
import re
from textarena_env import TextArenaAction
from textarena_env.rewards import extract_feedback_counts, extract_guess, extract_wordle_feedback
from trl.experimental.openenv import generate_rollout_completions

def rollout_once(trainer, env, tokenizer, dataset_prompt, system_prompt, max_turns, max_new_tokens):
    result = env.reset()
    observation = result.observation

    prompt_ids = []
    completion_ids = []
    logprobs = []
    env_mask = []  # 1 for model-generated tokens, 0 for environment tokens
    model_outputs = []
    raw_rewards = []
    position_scores = []
    correct_scores = []
    prev_env_output_len = 0  # Track length to only add NEW portion each turn

    accumulated_messages: list[dict[str, str]] = [{"role": "system", "content": system_prompt}]
    # Build initial prompt (only once, at the start)
    # The initial env messages are included in the prompt, not completion
    base_prompt = observation.prompt or dataset_prompt
    initial_user_prompt = make_user_prompt(base_prompt, observation.messages)
    # Track initial env output length so we don't add it again
    initial_env_output = format_history(observation.messages) if observation.messages else ""
    prev_env_output_len = len(initial_env_output)
    initial_messages = accumulated_messages + [{"role": "user", "content": initial_user_prompt}]
    initial_prompt_text = tokenizer.apply_chat_template(
        initial_messages,
        add_generation_prompt=True,
        tokenize=False,
        enable_thinking=False,
    )
    # Tokenize initial prompt once - this is the base prompt for the entire episode.
    # GRPO expects one prompt-completion pair per episode, where:
    # - prompt_ids = the initial/base prompt (what the model sees at episode start)
    # - completion_ids = all model responses + env feedback from all turns concatenated
    # Note: The actual prompts used for generation in each turn are longer (include conversation history),
    # but we only count the initial prompt tokens here.
    initial_prompt_ids = tokenizer.encode(initial_prompt_text, add_special_tokens=False)
    prompt_ids.extend(initial_prompt_ids)

    for _turn in range(max_turns):
        if result.done:
            break

        base_prompt = observation.prompt or dataset_prompt
        user_prompt = make_user_prompt(base_prompt, observation.messages)
        messages = accumulated_messages + [{"role": "user", "content": user_prompt}]
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False,
        )

        rollout_outputs = generate_rollout_completions(
            trainer, [prompt_text], generation_overrides={"max_tokens": max_new_tokens}
        )[0]
        # Add model-generated completion tokens and logprobs with newlines for readability
        newline_tokens = tokenizer.encode("\n", add_special_tokens=False)
        completion_ids.extend(newline_tokens)  # newline before guess
        logprobs.extend([0.0] * len(newline_tokens))
        env_mask.extend([1] * len(newline_tokens))  # newlines are part of model output format

        completion_ids.extend(rollout_outputs["completion_ids"])
        logprobs.extend(rollout_outputs["logprobs"])
        env_mask.extend([1] * len(rollout_outputs["completion_ids"]))  # model-generated tokens

        completion_ids.extend(newline_tokens)  # newline after guess
        logprobs.extend([0.0] * len(newline_tokens))
        env_mask.extend([1] * len(newline_tokens))  # newlines are part of model output format
        completion_text = rollout_outputs.get("text") or tokenizer.decode(
            rollout_outputs["completion_ids"], skip_special_tokens=True
        )
        guess = extract_guess(completion_text)
        model_outputs.append(completion_text.strip())  # Store raw model output for format reward

        result = env.step(TextArenaAction(message=guess))

        raw_rewards.append(float(result.reward or 0.0))
        observation = result.observation
        correct_score = float(result.reward or 0.0)
        feedback = extract_wordle_feedback(observation)

        full_env_output = format_history(observation.messages) if observation.messages else ""
        new_env_output = full_env_output[prev_env_output_len:].lstrip("\n")
        prev_env_output_len = len(full_env_output)

        if new_env_output:
            env_output_tokens = tokenizer.encode(new_env_output, add_special_tokens=False)
            completion_ids.extend(env_output_tokens)  # Add to completion_ids
            logprobs.extend([0.0] * len(env_output_tokens))  # Placeholder (ignored via env_mask=0)
            env_mask.extend([0] * len(env_output_tokens))  # Environment tokens - mask out from loss
            completion_with_env = completion_text + "\n" + new_env_output
        else:
            completion_with_env = completion_text

        accumulated_messages.append({"role": "user", "content": user_prompt})
        accumulated_messages.append({"role": "assistant", "content": completion_with_env})

        if not feedback:
            position_score = 0.0
        else:
            green_count, yellow_count = extract_feedback_counts(feedback)
            position_score = (green_count + 0.5 * yellow_count) / 5.0

        position_scores.append(position_score)
        correct_scores.append(correct_score)

    # Use the final correct reward (win/lose is binary at end)
    correct_reward_value = correct_scores[-1] if correct_scores else (raw_rewards[-1] if raw_rewards else 0.0)

    # Position reward as shaping signal:
    # - If model WINS: position_reward = 1.0 (no penalty for winning fast)
    # - If model LOSES: position_reward = last attempt (where it ended up)
    if correct_reward_value >= 1.0:
        final_position_reward = 1.0
    else:
        final_position_reward = position_scores[-1] if position_scores else 0.0

    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
        "env_mask": env_mask,
        "raw_rewards": raw_rewards,
        "correct_reward": correct_reward_value,
        "position_reward": final_position_reward,
        "model_outputs": model_outputs,
    }

### Helper functions

Supporting utilities used in `rollout_once`:

- **`make_user_prompt`**: builds the user prompt combining the conversation history.
- **`format_history`**: formats the conversation log for consistent context.

In [11]:
# @title Helpers definition (click to expand)
def format_history(messages) -> str:
    lines = []
    for message in messages:
        tag = message.category or "MESSAGE"
        content = message.content.strip()
        if not content:
            continue
        lines.append(f"[{tag}] {content}")
    return "\n".join(lines)


def make_user_prompt(prompt_text, messages) -> str:
    history = format_history(messages)
    # Only use messages for conversation history - the prompt is already included as the first message
    history_section = history if history else "[PROMPT] Awaiting first feedback."
    return f"Conversation so far:\n{history_section}\n\nReply with your next guess enclosed in square brackets."

## Define reward functions

To guide the agent's learning process, we define simple reward functions that map the feedback from the environment into numeric signals.  
Each function corresponds to a specific aspect of the **Wordle** game:

- ✅ **`reward_correct`**: rewards the model when it guesses the correct word (binary: 0 or 1).  
- 🎯 **`reward_position`**: rewards progress based on letter feedback. Green letters worth 1.0, yellow worth 0.5, normalized by 5. If the model wins, this is set to 1.0.
- 📝 **`reward_format_strict`**: rewards correct output format `[xxxxx]`. Returns proportion of correctly formatted outputs across all turns.

These functions return lists of float values that the **GRPOTrainer** uses during optimization.  
By combining them, the model learns to balance correctness, information gathering, and proper formatting in its guessing strategy.

In [12]:
def reward_correct(completions, **kwargs):
    """Reward from environment (correct answer)."""
    rewards = kwargs.get("correct_reward") if kwargs else None
    if rewards is None:
        return [0.0 for _ in completions]
    return [float(r) for r in rewards]


def reward_position(completions, **kwargs):
    """Position reward: green worth 1.0, yellow worth 0.5, normalized by 5."""
    rewards = kwargs.get("position_reward") if kwargs else None
    if rewards is None:
        return [0.0 for _ in completions]
    return [float(r) for r in rewards]


def compute_format_reward(model_outputs):
    """Compute format reward from a list of model outputs (one per turn).

    Each output should be exactly [5 letters] with optional whitespace.
    Returns proportion of correctly formatted outputs.
    """
    if not model_outputs:
        return 0.0

    exact_pattern = re.compile(r"^\s*\[[A-Za-z]{5}\]\s*$")
    correct_count = sum(1 for output in model_outputs if exact_pattern.match(output))

    return correct_count / len(model_outputs)


def reward_format_strict(completions, **kwargs):
    """Format reward - pre-computed in rollout_func."""
    rewards = kwargs.get("format_reward") if kwargs else None
    if rewards is None:
        return [0.0 for _ in completions]
    return [float(r) for r in rewards]

## Create dataset

We create a dataset with repeated prompts to control the number of training episodes.  
Each entry in the dataset triggers one rollout episode during training. The `dataset_prompt` provides the initial instruction to the model before each game starts.

In [13]:
from datasets import Dataset

dataset_size = 3000
dataset_prompt = "Play Wordle like an expert."

dataset = Dataset.from_dict({"prompt": [dataset_prompt] * dataset_size})

## Set GRPO Config

Next, we define the **GRPOConfig**, which controls all key training parameters.  
This configuration specifies how the model interacts with **vLLM**, manages memory, and logs results.

In [14]:
from trl import GRPOConfig

grpo_config = GRPOConfig(
    # Training schedule / optimization
    num_train_epochs = 1,                     # Number of full dataset passes
    learning_rate = 1e-6,                     # Learning rate for the optimizer
    gradient_accumulation_steps = 1,         # Accumulate gradients over multiple steps
    per_device_train_batch_size = 64,          # Batch size per GPU (number of prompts processed together)
    warmup_steps = 10,                        # Steps for learning rate warmup
    optim="adamw_8bit",                      # Optimizer
    max_grad_norm=1.0,                        # Clip gradients to prevent explosion

    # GRPO configuration
    num_generations = 4,                      # Number of rollout episodes per prompt (for variance reduction)
    max_completion_length=1024,               # Full episode length, not per-turn
    log_completions = False,                  # Log completions for debugging

    # Logging / reporting
    output_dir = 'outputs',                  # Directory for checkpoints and logs
    report_to="trackio",                      # Experiment tracking tool (integrates with HF Spaces)
    trackio_space_id = 'outputs',            # HF Space where experiment tracking will be saved
    logging_steps = 1,                        # Log metrics every N steps
    save_steps = 10,                          # Interval for saving checkpoints

    gradient_checkpointing = True,            # Enable activation recomputation to save memory
)

## Create `GRPOTrainer` and start training

Now we initialize the `GRPOTrainer`, which manages the entire reinforcement learning loop.

It takes the model, tokenizer, reward functions, rollout function, and dataset defined earlier.  
The trainer coordinates the interaction between the model and the environment, applies the reward signals, and updates the policy.

Finally, we call `trainer.train()` to start the fine-tuning process and let the model learn to play Wordle through feedback and iteration.

In [15]:
import sys
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

In [16]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        reward_correct,
        reward_position,
        reward_format_strict,
    ],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
)

/home/datta0/unsloth_nbs/nb/unsloth_compiled_cache/UnslothGRPOTrainer.py:4787: UserWarning: You are using 'rollout_func', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  super().__init__(


Show memory stats before training

In [17]:
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA B200. Max memory = 178.353 GB.
89.078 GB of memory reserved.


And train!

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 1 | Total steps = 187
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 1 x 1) = 64
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


* Trackio project initialized: huggingface
* Trackio metrics will be synced to Hugging Face Dataset: imdatta0/outputs-dataset
* Found existing space: https://huggingface.co/spaces/imdatta0/outputs
* View dashboard by going to: https://imdatta0-outputs.hf.space/


[unsloth_zoo.log|INFO]Unsloth: num_items_in_batch = None


* NVIDIA GPU detected, enabling automatic GPU metrics logging
* Created new run: imdatta0-1773138781


/home/datta0/repos/vllm/vllm/v1/attention/backends/flashinfer.py:908: DeprecationWarning: 
    Prefer using device seq_lens directly to avoid implicit H<>D sync.
    If a CPU copy is needed, use `seq_lens.cpu()` instead.
    Will be removed in a future release, please migrate as soon as possible.
    
  seq_lens_cpu = common_attn_metadata.seq_lens_cpu if needs_seq_lens_cpu else None


Show memory stats after training

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_training = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
training_memory_percentage = round(used_memory_for_training / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_training} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {training_memory_percentage} %.")

In [ ]:
env.close()
trainer.save_model(output_dir)
trainer.push_to_hub()

## Load the Fine-Tuned Model and Run Inference

Now let's test our fine-tuned model by loading the **adapter** and running **inference**.  
We begin by loading the **base model**, attaching the adapter, and obtaining the final fine-tuned model ready for evaluation.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "sergiopaniego/wordle-grpo-Qwen3-1.7B" # Replace with your HF username or organization

fine_tuned_model = AutoModelForCausalLM.from_pretrained(model_name, dtype="float32", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

Now that we have the fine-tuned model loaded, we can start playing Wordle.  
To make this easier, we'll define a reusable function so we can play multiple rounds.  
The function implements the same logic we explored earlier.

In [ ]:
MAX_TURNS=6

def play_wordle(env, model, tokenizer):
    result = env.reset()
    observation = result.observation

    print("📜 Initial Prompt:\n" + observation.prompt)

    for turn in range(MAX_TURNS):
        if result.done:
            break

        user_prompt = make_user_prompt(observation.prompt, observation.messages)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False,
        )

        model_inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

        # Decode and extract model response
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
        guess = extract_guess(generated_text)

        print(f"\n🎯 Turn {turn}: model replied with -> {generated_text}")
        print(f"   Parsed guess: {guess}")

        result = env.step(TextArenaAction(message=guess))
        observation = result.observation

        print("   Feedback messages:")
        for message in observation.messages:
            print(f"     [{message.category}] {message.content}")

    print("\n✅ Game finished")
    print(f"   Reward: {result.reward}")
    print(f"   Done: {result.done}")

Let's play the game!

In [ ]:
try:
    play_wordle(env, fine_tuned_model, tokenizer)
finally:
    env.close()